In [1]:
import anndata as ad
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from scipy import stats
from datetime import datetime

In [ ]:
# "indir" is a custom input path, and "outdir" is a custom output path.
indir = '../03-stratified_RankSumTest_1vsOthers'
outdir = '../03-stratified_RankSumTest_1vsOthers'

In [3]:
# get the p-value results

In [4]:
levels = ["01_subclasses_in_neuron_1vsOthers", "02_subclasses_in_NN_1vsOthers"]

In [12]:
for level_ in tqdm(levels):
    print(level_)
    df_5mCG = pd.concat([ad.read_h5ad(f'{indir}/{level_}/result/5mCG_RankSumTest_result_chr{chr_number}.h5ad').to_df() for chr_number in range(1,20,1)], axis=1)
    df_5hmCG = pd.concat([ad.read_h5ad(f'{indir}/{level_}/result/5hmCG_RankSumTest_result_chr{chr_number}.h5ad').to_df() for chr_number in range(1,20,1)], axis=1)
    df_5mCG_pvalue = df_5mCG.loc[df_5mCG.index.str.contains("pvalue")]
    df_5hmCG_pvalue = df_5hmCG.loc[df_5hmCG.index.str.contains("pvalue")]
    ad.AnnData(df_5mCG_pvalue).write_h5ad(f'{indir}/{level_}/5mCG_{'_'.join(level_.split('_')[-2:])}_pvalue.h5ad')
    ad.AnnData(df_5hmCG_pvalue).write_h5ad(f'{indir}/{level_}/5hmCG_{'_'.join(level_.split('_')[-2:])}_pvalue.h5ad')

  0%|          | 0/2 [00:00<?, ?it/s]

01_subclasses_in_neuron_1vsOthers
02_subclasses_in_NN_1vsOthers


In [17]:
# get the subclass diff from the robust mean

In [18]:
def robust_mean(series_, cutoff=0.25):
    if cutoff>0.5:
        cutoff=1-cutoff
    series_=series_[~np.isnan(series_)]
    total_len=len(series_)
    sorted_series_=sorted(series_)
    return np.mean(sorted_series_[
        int(np.floor(total_len*cutoff)):
        int(np.ceil(total_len*(1-cutoff)))])

In [ ]:
df_5mCG_mean = ad.read_h5ad('./04-summarize_RankSumTest/00-merged_allc_mcds/5mC_merged_allc_with_5mC_segment.h5ad').to_df()
df_5hmCG_mean = ad.read_h5ad('./04-summarize_RankSumTest/00-merged_allc_mcds/5hmC_merged_allc_with_5hmC_segment.h5ad').to_df()

In [21]:
df_5mCG_neuron_mean = df_5mCG_mean.loc[[i for i in df_5mCG_mean.index if "NN" not in i]]
df_5hmCG_neuron_mean = df_5hmCG_mean.loc[[i for i in df_5hmCG_mean.index if "NN" not in i]]

In [22]:
robust_mean_5mCG_neuron = df_5mCG_neuron_mean.apply(robust_mean,axis=0)

/share/home/renlh/miniconda3/envs/default/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/share/home/renlh/miniconda3/envs/default/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [23]:
robust_mean_5hmCG_neuron = df_5hmCG_neuron_mean.apply(robust_mean,axis=0)

In [24]:
robust_mean_5mCG_neuron

segment
chr1_3000826_3001630       0.974696
chr1_3003225_3003583       0.978536
chr1_3003720_3004531       0.965746
chr1_3007429_3007581       0.885332
chr1_3008544_3010494       0.970563
                             ...   
chr19_61323671_61324185    0.966713
chr19_61325280_61325656    0.943184
chr19_61326706_61327126    0.951537
chr19_61330072_61330238    0.955079
chr19_61330510_61331129    0.953756
Length: 2103205, dtype: float64

In [25]:
robust_mean_5hmCG_neuron

segment
chr1_3000826_3001630       0.091597
chr1_3003225_3003380       0.151314
chr1_3003720_3003899       0.211662
chr1_3004529_3006188       0.110922
chr1_3006415_3007171       0.220980
                             ...   
chr19_61325147_61326748    0.058885
chr19_61326952_61328153    0.084004
chr19_61329973_61330074    0.214776
chr19_61330083_61330182    0.087737
chr19_61330464_61330837    0.196274
Length: 1674096, dtype: float64

In [ ]:
ad.AnnData(df_5mCG_neuron_mean - robust_mean_5mCG_neuron).write_h5ad("../03-stratified_RankSumTest_1vsOthers/01_subclasses_in_neuron_1vsOthers/5mCG_neuron_frac_segment_diff.h5ad")

In [ ]:
ad.AnnData(df_5hmCG_neuron_mean - robust_mean_5hmCG_neuron).write_h5ad("../03-stratified_RankSumTest_1vsOthers/01_subclasses_in_neuron_1vsOthers/5hmCG_neuron_frac_segment_diff.h5ad")

In [29]:
df_5mCG_NN_mean = df_5mCG_mean.loc[[i for i in df_5mCG_mean.index if "NN" in i]]
df_5hmCG_NN_mean = df_5hmCG_mean.loc[[i for i in df_5hmCG_mean.index if "NN" in i]]

In [30]:
robust_mean_5mCG_NN = df_5mCG_NN_mean.apply(robust_mean,axis=0)

/share/home/renlh/miniconda3/envs/default/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/share/home/renlh/miniconda3/envs/default/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [31]:
robust_mean_5hmCG_NN = df_5hmCG_NN_mean.apply(robust_mean,axis=0)

In [32]:
robust_mean_5mCG_NN

segment
chr1_3000826_3001630       0.908626
chr1_3003225_3003583       0.923468
chr1_3003720_3004531       0.877611
chr1_3007429_3007581       0.737861
chr1_3008544_3010494       0.850127
                             ...   
chr19_61323671_61324185    0.888018
chr19_61325280_61325656    0.833735
chr19_61326706_61327126    0.911456
chr19_61330072_61330238    0.948993
chr19_61330510_61331129    0.943983
Length: 2103205, dtype: float64

In [33]:
robust_mean_5hmCG_NN

segment
chr1_3000826_3001630       0.021647
chr1_3003225_3003380       0.026994
chr1_3003720_3003899       0.024661
chr1_3004529_3006188       0.017763
chr1_3006415_3007171       0.059983
                             ...   
chr19_61325147_61326748    0.009487
chr19_61326952_61328153    0.013599
chr19_61329973_61330074    0.058153
chr19_61330083_61330182    0.034947
chr19_61330464_61330837    0.052591
Length: 1674096, dtype: float64

In [ ]:
ad.AnnData(df_5mCG_NN_mean - robust_mean_5mCG_NN).write_h5ad("../03-stratified_RankSumTest_1vsOthers/02_subclasses_in_NN_1vsOthers/5mCG_NN_frac_segment_diff.h5ad")

In [ ]:
ad.AnnData(df_5hmCG_NN_mean - robust_mean_5hmCG_NN).write_h5ad("../03-stratified_RankSumTest_1vsOthers/02_subclasses_in_NN_1vsOthers/5hmCG_NN_frac_segment_diff.h5ad")

In [5]:
# perserve segments at least 200bp

In [6]:
def get_segment_length(segment_):
    chrom_, start_, end_ = segment_.split("_")
    return int(end_) - int(start_)

In [7]:
def adjust_p_value_withNaN(p_values):
    """for pd.Series p_values input"""
    # Step 1: Identify non-NaN indices
    valid_indices = ~np.isnan(p_values)
    valid_p_values = p_values[valid_indices]
    # Step 2: Perform BH correction on non-NaN values
    corrected_p_values=stats.false_discovery_control(valid_p_values, method="bh")
    # Step 3: Reinsert corrected values into the original array
    adjusted_p_values = np.full_like(p_values, np.nan)  # Start with an array of NaN
    adjusted_p_values[valid_indices] = corrected_p_values
    return(pd.Series(adjusted_p_values, index=p_values.index))

In [8]:
allc_name=lambda cellID : "allc_" + cellID

In [ ]:
for modification_ in tqdm(["5mC", "5hmC"]):
    print("****"+modification_+"****")
    print(f'{datetime.now()}\t Started reading {modification_}G_frac in the format of segment_by_cell ..')
    data_list = []
    for chr_number in range(1,20,1):
        print(f'{datetime.now()}\t  Starting chr{chr_number}..')
        data_chr = ad.read_h5ad(f'../02-generate_segment_MCDS/02.segment_mcds/{modification_[1:]}G_frac_segment_by_cell_chr{chr_number}.h5ad').to_df()
        data_columns_1 = pd.Series(data_chr.columns)
        data_columns_2 = data_columns_1[data_columns_1.apply(get_segment_length)>=200]
        data_chr = data_chr[data_columns_2]
        data_list.append(data_chr)
    data = pd.concat(data_list, axis=1)
    print(f'{datetime.now()}\t Finished reading {modification_}G_frac in the format of segment_by_cell ..')
    
    data_count = data.count(axis=0)
    print(f'{datetime.now()}\t Calculation of data_count finished ..\n')
    
    for level_ in tqdm(levels):
        print("-----"+level_+"-----")
        print(f'{datetime.now()}\t Started reading p-values')
        original_pvalue_df= ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{'_'.join(level_.split('_')[-2:])}_pvalue.h5ad').to_df()
        print(f'{datetime.now()}\t Finished reading p-values')

        label_ = "subclass_label"
        subclass_list=pd.read_csv('../../../03.data/04.config_files/subclass_order.csv', header=None).loc[:, 0].to_list()
        if level_ == "01_subclasses_in_neuron_1vsOthers":
            subclass_list = [i for i in subclass_list if "NN" not in i]
        elif level_ == "02_subclasses_in_NN_1vsOthers":
            subclass_list = [i for i in subclass_list if "NN" in i]
        print(f'{datetime.now()}\t Started calculating mask df to mask cases where CellNumber<10 ..')
        
        meta_data = pd.read_csv('TSO-joint.DNA_QC_stat.young.add_celltype.csv', header=0)
        meta_data2 = meta_data[meta_data["total_QC"]==1]
        meta_data3 = meta_data2[[label_, modification_[1:]+'_SampleID']]
        meta_data_group = meta_data3.groupby(label_)
        data_dict_count = {subclass_ : data.loc[allc_name(meta_data_group.get_group(subclass_)[modification_[1:]+'_SampleID'])].count(axis=0) \
                   for subclass_ in subclass_list}
        data_count_df1=pd.DataFrame(data_dict_count).T
        data_count_df2=data_count_df1.loc[subclass_list,:]
        data_count_df3=(data_count_df2>=10)*((data_count-data_count_df2)>=10)
        print(f'{datetime.now()}\t Finished calculating mask df to mask cases where CellNumber<10 ..')
        mask_df = data_count_df3.map(lambda x: np.float32(np.nan) if not x else np.float32(x))
        # data_count_df3 is of dtype bool, mannually asign np.float32 to avoid mask_df acquiring dtypes of "object"
        print(list(zip(mask_df.index, original_pvalue_df.index)))
        mask_df.index = original_pvalue_df.index

        masked_pvalue_df = original_pvalue_df * mask_df
        print(f'{datetime.now()}\t Finished masking ..')
        adjusted_masked_pvalue_df = masked_pvalue_df.apply(adjust_p_value_withNaN, axis=1)
        print(f'{datetime.now()}\t Finished BH adjusting ..')
        ad.AnnData(adjusted_masked_pvalue_df).write_h5ad(f'{outdir}/{level_}/{modification_}G_{level_.split("_")[-2]}_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad')
        print(f'{datetime.now()}\t Saved {modification_}G_{level_.split("_")[-2]}_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad\n\n')

  0%|          | 0/2 [00:00<?, ?it/s]

****5mC****
2025-11-17 18:36:31.108019	 Started reading 5mCG_frac in the format of segment_by_cell ..
2025-11-17 18:36:31.108030	  Starting chr1..
2025-11-17 18:37:02.679616	  Starting chr2..
2025-11-17 18:37:34.007789	  Starting chr3..
2025-11-17 18:38:02.057356	  Starting chr4..
2025-11-17 18:38:31.843925	  Starting chr5..
2025-11-17 18:39:01.187146	  Starting chr6..
2025-11-17 18:39:28.338074	  Starting chr7..
2025-11-17 18:39:55.724354	  Starting chr8..
2025-11-17 18:40:21.590109	  Starting chr9..
2025-11-17 18:40:46.680431	  Starting chr10..
2025-11-17 18:41:12.794389	  Starting chr11..
2025-11-17 18:41:38.240171	  Starting chr12..
2025-11-17 18:42:08.925633	  Starting chr13..
2025-11-17 18:42:33.067209	  Starting chr14..
2025-11-17 18:42:59.281344	  Starting chr15..
2025-11-17 18:43:26.396969	  Starting chr16..
2025-11-17 18:43:48.003265	  Starting chr17..
2025-11-17 18:43:59.141227	  Starting chr18..
2025-11-17 18:44:20.518489	  Starting chr19..
2025-11-17 18:51:48.423086	 Finis

  0%|          | 0/2 [00:00<?, ?it/s]

-----01_subclasses_in_neuron_1vsOthers-----
2025-11-17 18:54:56.922131	 Started reading p-values
2025-11-17 18:54:58.760864	 Finished reading p-values
2025-11-17 18:54:58.763018	 Started calculating mask df to mask cases where CellNumber<10 ..
2025-11-17 19:12:00.098856	 Finished calculating mask df to mask cases where CellNumber<10 ..
[('L2/3 IT CTX Glut', 'L2/3 IT CTX Glut_Others_pvalue'), ('L4/5 IT CTX Glut', 'L4/5 IT CTX Glut_Others_pvalue'), ('L5 IT CTX Glut', 'L5 IT CTX Glut_Others_pvalue'), ('L6 IT CTX Glut', 'L6 IT CTX Glut_Others_pvalue'), ('IT AON-TT-DP Glut', 'IT AON-TT-DP Glut_Others_pvalue'), ('LA-BLA-BMA-PA Glut', 'LA-BLA-BMA-PA Glut_Others_pvalue'), ('L2/3 IT RSP Glut', 'L2/3 IT RSP Glut_Others_pvalue'), ('L4 RSP-ACA Glut', 'L4 RSP-ACA Glut_Others_pvalue'), ('L5 ET CTX Glut', 'L5 ET CTX Glut_Others_pvalue'), ('SUB-ProS Glut', 'SUB-ProS Glut_Others_pvalue'), ('CA1-ProS Glut', 'CA1-ProS Glut_Others_pvalue'), ('CA3 Glut', 'CA3 Glut_Others_pvalue'), ('CLA-EPd-CTX Car3 Glut',

  0%|          | 0/2 [00:00<?, ?it/s]

-----01_subclasses_in_neuron_1vsOthers-----
2025-11-17 19:28:46.687365	 Started reading p-values
2025-11-17 19:28:47.844662	 Finished reading p-values
2025-11-17 19:28:47.846081	 Started calculating mask df to mask cases where CellNumber<10 ..
2025-11-17 19:34:05.657512	 Finished calculating mask df to mask cases where CellNumber<10 ..
[('L2/3 IT CTX Glut', 'L2/3 IT CTX Glut_Others_pvalue'), ('L4/5 IT CTX Glut', 'L4/5 IT CTX Glut_Others_pvalue'), ('L5 IT CTX Glut', 'L5 IT CTX Glut_Others_pvalue'), ('L6 IT CTX Glut', 'L6 IT CTX Glut_Others_pvalue'), ('IT AON-TT-DP Glut', 'IT AON-TT-DP Glut_Others_pvalue'), ('LA-BLA-BMA-PA Glut', 'LA-BLA-BMA-PA Glut_Others_pvalue'), ('L2/3 IT RSP Glut', 'L2/3 IT RSP Glut_Others_pvalue'), ('L4 RSP-ACA Glut', 'L4 RSP-ACA Glut_Others_pvalue'), ('L5 ET CTX Glut', 'L5 ET CTX Glut_Others_pvalue'), ('SUB-ProS Glut', 'SUB-ProS Glut_Others_pvalue'), ('CA1-ProS Glut', 'CA1-ProS Glut_Others_pvalue'), ('CA3 Glut', 'CA3 Glut_Others_pvalue'), ('CLA-EPd-CTX Car3 Glut',

In [10]:
np.sum(np.sum(masked_pvalue_df.isnull()))

/share/home/renlh/miniconda3/envs/default/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


67161

In [11]:
np.sum(np.sum(original_pvalue_df.isnull()))

27973

In [12]:
np.sum(np.sum(mask_df.isnull()))

67161

In [ ]:
for level_ in tqdm(levels):
    for modification_ in tqdm(["5mC", "5hmC"]):
        original_diff_df=ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{level_.split("_")[-2]}_frac_segment_diff.h5ad').to_df()
        # perserve only segements with lengths that are at least 200bp
        data_columns_1=pd.Series(original_diff_df.columns)
        data_columns_2=data_columns_1[data_columns_1.apply(get_segment_length)>=200]
        new_diff_df=original_diff_df[data_columns_2]
        ad.AnnData(new_diff_df).write_h5ad(f'{outdir}/{level_}/{modification_}G_{level_.split("_")[-2]}_frac_segment_diff.h5ad')

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [18]:
for modification_ in tqdm(["5mC", "5hmC"]):
    print("****"+modification_+"****")
    if modification_ == "5mC":
        diff_threshold_ = 0.3
    elif modification_ == "5hmC":
        diff_threshold_ = 0.2
    else:
        raise Exception("wrong modification")
        
    for level_ in tqdm(levels):
        print("-----"+level_+"-----")
        original_diff_df=ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{level_.split("_")[-2]}_frac_segment_diff.h5ad').to_df()
        # perserve only segements with lengths that are at least 200bp
        data_columns_1=pd.Series(original_diff_df.columns)
        data_columns_2=data_columns_1[data_columns_1.apply(get_segment_length)>=200]
        df_diff = original_diff_df[data_columns_2]
        
        df_pvalue=ad.read_h5ad(f'{indir}/{level_}/{modification_}G_{level_.split("_")[-2]}_RankSumTest1vsOthers_AdjustedMasked_pvalue.h5ad').to_df()
        the_index=pd.Series(df_pvalue.index).apply(lambda x : x.replace("_Others_pvalue", ""))
        df_pvalue.index=the_index
        df_diff.index=the_index
        for pvalue_threshold_ in [0.05]:
            df_DMR_states=np.sign(df_diff) * (df_diff.abs()>diff_threshold_) * (df_pvalue <= pvalue_threshold_)
            print(modification_, level_, pvalue_threshold_)
            print(np.sum(df_DMR_states.abs().sum(axis=0)>0))
            ad.AnnData(df_DMR_states).write_h5ad(f'{outdir}/{level_}/{modification_}G_DMR_states_{level_.split("_")[-2]}_1vsOthers.h5ad')

  0%|          | 0/2 [00:00<?, ?it/s]

****5mC****


  0%|          | 0/2 [00:00<?, ?it/s]

-----01_subclasses_in_neuron_1vsOthers-----
5mC 01_subclasses_in_neuron_1vsOthers 0.05
131352
-----02_subclasses_in_NN_1vsOthers-----
5mC 02_subclasses_in_NN_1vsOthers 0.05
57108
****5hmC****


  0%|          | 0/2 [00:00<?, ?it/s]

-----01_subclasses_in_neuron_1vsOthers-----
5hmC 01_subclasses_in_neuron_1vsOthers 0.05
200438
-----02_subclasses_in_NN_1vsOthers-----
5hmC 02_subclasses_in_NN_1vsOthers 0.05
9357
